# E7: PhoBERT-base-v2 + classification head

Phân loại bình luận tiếng Việt của sinh viên PTIT thành **Toxic (1)** và **Non-toxic (0)**.

| Mã | Mô hình | Đầu vào |
|---|---|---|
| E7 | `vinai/phobert-base-v2` + classification head | `message_segmented` (tách từ `message_clean` bằng underthesea, tạo sẵn ở local) |

**Quy trình:** đọc dữ liệu → gắn `message_segmented` → tìm siêu tham số theo 2 giai đoạn (early stopping và chọn mô hình theo F1 Toxic trên Validation) → đánh giá Validation → đánh giá Test **một lần** → phân tích lỗi → lưu config, checkpoint, tham số và log huấn luyện.

## Cách chạy trên Kaggle (Internet OFF)
1. **Add Input** các dataset:
   - `train-nlp`, `val-nlp`, `test-npm`: 3 file xlsx.
   - Dataset chứa **PhoBERT-base-v2**: thư mục có `config.json`, file trọng số (`model.safetensors` hoặc `pytorch_model.bin`), `vocab.txt`, `bpe.codes`.
   - Dataset chứa **`message_segmented.csv`**, được tạo bằng `python -X utf8 scripts/segment_underthesea.py` ở local.
2. Trong **Settings**: chọn Accelerator **GPU** và tắt Internet.
3. Kiểm tra các đường dẫn trong ô CONFIG. Để `MODEL_PATH` hoặc `SEGMENTED_PATH` là `None` thì notebook tự tìm trong `/kaggle/input`.
4. Đặt `QUICK_RUN = True` rồi **Run All** để kiểm tra pipeline (vài phút).
5. Đặt lại `QUICK_RUN = False`, chọn **Save Version → Save & Run All** (ước tính 1–2 giờ).
6. Kết quả nằm trong `/kaggle/working/E7/` (tab **Output**).

## 1. Cấu hình

In [ ]:
# ============================== CONFIG ==============================
EXP_ID = "E7"
MODEL_NAME = "vinai/phobert-base-v2"   # tên gốc, chỉ để ghi nhận; trọng số nạp offline từ MODEL_PATH
MODEL_PATH = '/kaggle/input/datasets/quctriunguyn/phobert-base-v2/phobert-base-v2'                      # thư mục PhoBERT đã upload; None = tự tìm thư mục có bpe.codes trong /kaggle/input
MODEL_MARKERS = ["bpe.codes"]          # file tokenizer dùng để nhận ra thư mục model
TEXT_COL = "message_segmented"         # cột đưa vào mô hình
SEGMENTED_PATH = '/kaggle/input/datasets/quctriunguyn/message-segmented/message_segmented.csv'                  # message_segmented.csv; None = tự tìm trong /kaggle/input

TRAIN_PATH = "/kaggle/input/datasets/quctriunguyn/train-nlp/train.xlsx"
VAL_PATH = "/kaggle/input/datasets/quctriunguyn/val-nlp/val.xlsx"
TEST_PATH = "/kaggle/input/datasets/quctriunguyn/test-npm/test.xlsx"

QUICK_RUN = False      # True: chạy thử pipeline (tập con 200/100/100, 1 cấu hình, 1 epoch)

SEED = 42
MAX_EPOCHS = 8         # số epoch tối đa; số epoch thực tế do early stopping quyết định
PATIENCE = 2           # dừng khi F1 Toxic trên val không tăng sau PATIENCE epoch liên tiếp
MICRO_BATCH = 16       # batch vật lý trên GPU; batch_size lớn hơn đạt được nhờ gradient accumulation
EVAL_BATCH = 64
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
GRAD_CHECKPOINTING = False   # bật nếu bị OOM

# Giai đoạn 1: lr × weighted loss (batch_size và max_len cố định)
SEARCH_STAGE1 = {"lr": [1e-5, 2e-5, 3e-5, 5e-5], "weighted": [False, True], "batch_size": [16], "max_len": [128]}
# Giai đoạn 2: batch_size × max_len, dùng lr và weighted tốt nhất của giai đoạn 1
SEARCH_STAGE2 = {"batch_size": [16, 32], "max_len": [128, 256]}

## 2. Môi trường và đường dẫn

In [ ]:
import os
os.environ["HF_HUB_OFFLINE"] = "1"        # Internet OFF: không gọi tới Hugging Face Hub
os.environ["TRANSFORMERS_OFFLINE"] = "1"

import importlib.util, sys
if importlib.util.find_spec("openpyxl") is None:
    raise ImportError("Thiếu openpyxl để đọc .xlsx; môi trường offline không pip install được.")

import contextlib, gc, glob, itertools, json, math, random, shutil, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import torch
import torch.nn.functional as F
import transformers
from IPython.display import display
from sklearn.metrics import (accuracy_score, average_precision_score, confusion_matrix,
                             f1_score, precision_recall_curve, precision_recall_fscore_support)
from torch.utils.data import DataLoader, Dataset
from transformers import (AutoConfig, AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, get_linear_schedule_with_warmup)

transformers.logging.set_verbosity_error()  # ẩn cảnh báo "classifier mới được khởi tạo" (là điều mong đợi)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
GPU_NAME = torch.cuda.get_device_name(0) if USE_AMP else "CPU"
print(f"python {sys.version.split()[0]} | torch {torch.__version__} | transformers {transformers.__version__} | device: {GPU_NAME}")
if not USE_AMP:
    print("CẢNH BÁO: không thấy GPU. Hãy bật Accelerator = GPU trong Settings của Kaggle.")


def find_file(path, filename):
    """Trả về path nếu tồn tại; nếu không, tìm filename trong /kaggle/input."""
    if path and os.path.isfile(path):
        return path
    hits = sorted(glob.glob(f"/kaggle/input/**/{filename}", recursive=True))
    if hits:
        print(f"Không thấy {path!r} -> dùng {hits[0]}")
        return hits[0]
    raise FileNotFoundError(f"Không tìm thấy {filename}. Kiểm tra Add Input và đường dẫn trong CONFIG.")


def find_model_dir(path, markers):
    """Thư mục chứa config.json và một file tokenizer trong markers.
    Tìm trong path (kể cả thư mục con), nếu không có thì tìm trong toàn bộ /kaggle/input."""
    roots = [p for p in [path, "/kaggle/input"] if p and os.path.isdir(p)]
    for root in roots:
        for d, _, files in os.walk(root):
            if "config.json" in files and any(m in files for m in markers):
                if path and not d.startswith(path):
                    print(f"Không thấy model trong {path!r} -> dùng {d}")
                return d
    raise FileNotFoundError(f"Không tìm thấy thư mục model (config.json + {markers}) trong {roots}. "
                            "Hãy Add Input dataset chứa model và sửa MODEL_PATH.")


TRAIN_PATH = find_file(TRAIN_PATH, "train.xlsx")
VAL_PATH = find_file(VAL_PATH, "val.xlsx")
TEST_PATH = find_file(TEST_PATH, "test.xlsx")
MODEL_DIR = find_model_dir(MODEL_PATH, MODEL_MARKERS)
print("Model:", MODEL_DIR, "->", sorted(os.listdir(MODEL_DIR)))

WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("outputs")
OUT_DIR = os.path.join(WORK_DIR, EXP_ID)
BEST_DIR = os.path.join(OUT_DIR, "best_model")
LOG_PATH = os.path.join(OUT_DIR, "train_log.csv")
os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR:", OUT_DIR)

if QUICK_RUN:
    MAX_EPOCHS = 1
    print("QUICK_RUN = True: chỉ để kiểm tra pipeline, KHÔNG dùng kết quả cho báo cáo.")

SEGMENTER = None  # tên công cụ tách từ (chỉ E7 dùng)


def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def amp_ctx():
    """fp16 autocast trên GPU; không làm gì trên CPU."""
    return torch.autocast("cuda", dtype=torch.float16) if USE_AMP else contextlib.nullcontext()


set_seed(SEED)

## 3. Dữ liệu

Các file đều dùng sheet `TRAIN_DATA` và mọi ô là chuỗi, nên đọc với `dtype=str` rồi ép `label` sang `int`.
Các dòng có `data_status == "review"` (nội dung rỗng hoặc chỉ có `?`) bị loại ở cả 3 tập.

In [ ]:
TEXT_COLS = ["post_message", "parent_message", "message_original", "message_clean", "message_normalized"]


def load_split(name, path):
    df = pd.read_excel(path, sheet_name="TRAIN_DATA", dtype=str)
    n_raw = len(df)
    df = df[df["data_status"].fillna("").str.strip() == "valid"].copy()
    n_valid = len(df)
    for c in TEXT_COLS:
        df[c] = df[c].fillna("").astype(str)
    df["label"] = df["label"].astype(int)
    assert set(df["label"].unique()) <= {0, 1}, "label phải là 0/1"
    if QUICK_RUN:
        df = df.sample(n=min(len(df), 200 if name == "train" else 100), random_state=SEED)
    df = df.reset_index(drop=True)
    print(f"{name:5s}: {n_raw} dòng, {n_valid} hợp lệ (bỏ {n_raw - n_valid} dòng review)"
          f"{f', dùng tập con {len(df)}' if QUICK_RUN else ''} | Toxic = {df['label'].mean():.2%}")
    return df


splits = {"train": load_split("train", TRAIN_PATH),
          "val": load_split("val", VAL_PATH),
          "test": load_split("test", TEST_PATH)}

## 4. Gắn cột message_segmented

PhoBERT cần đầu vào đã tách từ, các âm tiết trong một từ nối bằng `_` (ví dụ `sinh_viên`).
Kaggle chạy với Internet tắt nên không tách từ ngay trong notebook được. Cột `message_segmented` được tạo sẵn ở local bằng `scripts/segment_underthesea.py` (tách từ `message_clean` bằng underthesea), rồi ghép vào đây theo `sample_id`.
Notebook báo lỗi nếu thiếu `sample_id`, hoặc nếu `message_clean` trong file tách từ khác với dữ liệu đang dùng.

In [ ]:
SEGMENTED_PATH = find_file(SEGMENTED_PATH, "message_segmented.csv")
seg_df = pd.read_csv(SEGMENTED_PATH, dtype=str, keep_default_na=False)
SEGMENTER = seg_df["segmenter"].iloc[0] if "segmenter" in seg_df else "không rõ"
seg_map = dict(zip(seg_df["sample_id"], seg_df["message_segmented"]))
src_map = dict(zip(seg_df["sample_id"], seg_df["message_clean"]))

for name, df in splits.items():
    missing = df.loc[~df["sample_id"].isin(set(seg_map)), "sample_id"].tolist()
    if missing:
        raise ValueError(f"{name}: {len(missing)} sample_id không có trong {SEGMENTED_PATH}, ví dụ {missing[:5]}")
    n_mismatch = int((df["sample_id"].map(src_map) != df["message_clean"]).sum())
    if n_mismatch:
        raise ValueError(f"{name}: {n_mismatch} dòng có message_clean khác file tách từ -> file được tạo từ dữ liệu khác")
    df["message_segmented"] = df["sample_id"].map(seg_map)
    print(f"{name}: đã gắn message_segmented cho {len(df)} dòng")

print("Công cụ tách từ:", SEGMENTER)
display(splits["train"][["message_clean", "message_segmented"]].head(8))

## 5. Tokenizer và độ dài đầu vào

Thống kê độ dài token trên train để chọn `max_len`. Các giá trị `max_len` vượt giới hạn của mô hình (PhoBERT: 256) bị loại khỏi không gian tìm kiếm.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
# Họ RoBERTa dành 2 vị trí đầu cho padding_idx -> độ dài tối đa = max_position_embeddings - 2
MODEL_MAX_LEN = AutoConfig.from_pretrained(MODEL_DIR).max_position_embeddings - 2
print(f"Tokenizer: {type(tokenizer).__name__} | độ dài tối đa của mô hình: {MODEL_MAX_LEN} token")

_example = splits["train"][TEXT_COL].iloc[0]
print("Ví dụ:", _example)
print("Token:", tokenizer.tokenize(_example)[:40])

_lens = np.array([len(ids) for ids in tokenizer(splits["train"][TEXT_COL].tolist())["input_ids"]])
print(f"Số token/mẫu trên train (kể cả token đặc biệt): p50={np.percentile(_lens, 50):.0f} "
      f"p95={np.percentile(_lens, 95):.0f} p99={np.percentile(_lens, 99):.0f} max={_lens.max()}")
for L in sorted({*SEARCH_STAGE1["max_len"], *SEARCH_STAGE2["max_len"]}):
    print(f"  max_len={L}: {(_lens > L).mean():.2%} mẫu bị cắt bớt")

SEARCH_STAGE1["max_len"] = [L for L in SEARCH_STAGE1["max_len"] if L <= MODEL_MAX_LEN]
SEARCH_STAGE2["max_len"] = [L for L in SEARCH_STAGE2["max_len"] if L <= MODEL_MAX_LEN]

## 6. Dataset và DataLoader

In [ ]:
class TextDataset(Dataset):
    def __init__(self, texts, labels, max_len):
        self.enc = tokenizer(list(texts), truncation=True, max_length=max_len)
        self.labels = [int(x) for x in labels]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.labels[i]
        return item


collator = DataCollatorWithPadding(tokenizer)  # padding động theo câu dài nhất trong batch
_ds_cache = {}


def make_loader(split, max_len, batch_size, shuffle):
    key = (split, max_len)
    if key not in _ds_cache:
        _ds_cache[key] = TextDataset(splits[split][TEXT_COL], splits[split]["label"], max_len)
    gen = torch.Generator().manual_seed(SEED) if shuffle else None
    return DataLoader(_ds_cache[key], batch_size=batch_size, shuffle=shuffle, collate_fn=collator, generator=gen)

## 7. Chỉ số đánh giá

Lớp dương là **Toxic = 1**. Chỉ số chọn mô hình: **F1 Toxic trên Validation**; nếu bằng nhau thì xét Macro-F1, rồi Recall Toxic.

In [ ]:
METRIC_KEYS = ["toxic_precision", "toxic_recall", "toxic_f1", "macro_f1", "accuracy", "pr_auc"]
HP_KEYS = ["lr", "weighted", "batch_size", "max_len"]


def compute_metrics(y_true, prob, thr=0.5):
    y_true = np.asarray(y_true).astype(int)
    prob = np.asarray(prob, dtype=float)
    pred = (prob >= thr).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y_true, pred, labels=[1], average=None, zero_division=0)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    has_both = 0 < y_true.sum() < len(y_true)
    return {
        "toxic_precision": float(p[0]), "toxic_recall": float(r[0]), "toxic_f1": float(f[0]),
        "macro_f1": float(f1_score(y_true, pred, average="macro", zero_division=0)),
        "accuracy": float(accuracy_score(y_true, pred)),
        "pr_auc": float(average_precision_score(y_true, prob)) if has_both else float("nan"),
        "threshold": float(thr), "tn": int(tn), "fp": int(fp), "fn": int(fn), "tp": int(tp),
    }


def selection_key(m):
    return (m["toxic_f1"], m["macro_f1"], m["toxic_recall"])


def result_key(r):
    return (r["val_toxic_f1"], r["val_macro_f1"], r["val_toxic_recall"])


@torch.no_grad()
def predict_proba(model, loader):
    """Trả về (xác suất lớp Toxic, loss cross-entropy trung bình không trọng số)."""
    model.eval()
    probs, loss_sum, n = [], 0.0, 0
    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        labels = batch.pop("labels")
        with amp_ctx():
            logits = model(**batch).logits
        logits = logits.float()
        loss_sum += F.cross_entropy(logits, labels, reduction="sum").item()
        n += labels.size(0)
        probs.append(torch.softmax(logits, dim=-1)[:, 1].cpu().numpy())
    return np.concatenate(probs), loss_sum / n

## 8. Huấn luyện một cấu hình

Vòng lặp huấn luyện tự viết (không dùng `Trainer`, để không phụ thuộc thay đổi API giữa các phiên bản `transformers`):
AdamW + linear warmup, fp16, gradient accumulation (`batch_size = MICRO_BATCH × accum`), weighted cross-entropy tùy chọn,
đánh giá val sau mỗi epoch và early stopping theo F1 Toxic. Mỗi lần chạy, nếu tốt hơn mọi lần chạy trước, checkpoint được lưu vào `best_model/`.

In [ ]:
def make_optimizer(model, lr):
    no_decay = ("bias", "LayerNorm.weight")
    groups = [
        {"params": [p for n, p in model.named_parameters() if not n.endswith(no_decay)], "weight_decay": WEIGHT_DECAY},
        {"params": [p for n, p in model.named_parameters() if n.endswith(no_decay)], "weight_decay": 0.0},
    ]
    return torch.optim.AdamW(groups, lr=lr)


def train_one(hp, run_id, global_best_key):
    """Huấn luyện 1 cấu hình; lưu checkpoint vào BEST_DIR nếu tốt hơn global_best_key."""
    set_seed(SEED)
    t_start = time.time()
    accum = max(1, hp["batch_size"] // MICRO_BATCH)
    micro = hp["batch_size"] // accum
    train_loader = make_loader("train", hp["max_len"], micro, shuffle=True)
    val_loader = make_loader("val", hp["max_len"], EVAL_BATCH, shuffle=False)

    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_DIR, num_labels=2, ignore_mismatched_sizes=True).to(DEVICE)
    if GRAD_CHECKPOINTING:
        model.gradient_checkpointing_enable()
    optimizer = make_optimizer(model, hp["lr"])
    total_steps = math.ceil(len(train_loader) / accum) * MAX_EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, int(WARMUP_RATIO * total_steps), total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

    class_weight = None
    if hp["weighted"]:
        counts = np.bincount(splits["train"]["label"], minlength=2)
        class_weight = torch.tensor(counts.sum() / (2 * counts), dtype=torch.float, device=DEVICE)
    loss_fn = torch.nn.CrossEntropyLoss(weight=class_weight)

    best = {"key": None, "epoch": 0, "metrics": None, "state": None}
    bad_epochs = 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        t0, loss_sum, n_seen = time.time(), 0.0, 0
        optimizer.zero_grad(set_to_none=True)
        for step, batch in enumerate(train_loader, 1):
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            labels = batch.pop("labels")
            with amp_ctx():
                logits = model(**batch).logits
            loss = loss_fn(logits.float(), labels)
            scaler.scale(loss / accum).backward()
            loss_sum += loss.item() * labels.size(0)
            n_seen += labels.size(0)
            if step % accum == 0 or step == len(train_loader):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()

        val_prob, val_loss = predict_proba(model, val_loader)
        m = compute_metrics(splits["val"]["label"], val_prob)
        row = {"run_id": run_id, **hp, "epoch": epoch, "train_loss": loss_sum / n_seen, "val_loss": val_loss,
               **{f"val_{k}": m[k] for k in METRIC_KEYS}, "epoch_time_s": round(time.time() - t0, 1)}
        pd.DataFrame([row]).to_csv(LOG_PATH, mode="a", header=not os.path.exists(LOG_PATH), index=False)
        print(f"  [{run_id}] epoch {epoch}: train_loss={row['train_loss']:.4f} val_loss={val_loss:.4f} | "
              f"F1_toxic={m['toxic_f1']:.4f} P={m['toxic_precision']:.4f} R={m['toxic_recall']:.4f} "
              f"macroF1={m['macro_f1']:.4f} ({row['epoch_time_s']:.0f}s)")

        if best["key"] is None or selection_key(m) > best["key"]:
            best.update(key=selection_key(m), epoch=epoch, metrics=m,
                        state={k: v.detach().to("cpu", copy=True) for k, v in model.state_dict().items()})
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f"  [{run_id}] early stopping sau epoch {epoch}")
                break

    is_new_best = global_best_key is None or best["key"] > global_best_key
    if is_new_best:
        model.load_state_dict(best["state"])
        shutil.rmtree(BEST_DIR, ignore_errors=True)
        model.save_pretrained(BEST_DIR)
        tokenizer.save_pretrained(BEST_DIR)
        print(f"  [{run_id}] -> tốt nhất hiện tại, đã lưu checkpoint (epoch {best['epoch']}) vào {BEST_DIR}")

    result = {"run_id": run_id, **hp, "best_epoch": best["epoch"], "epochs_run": epoch,
              **{f"val_{k}": best["metrics"][k] for k in METRIC_KEYS},
              "time_min": round((time.time() - t_start) / 60, 1), "saved_as_best": is_new_best}
    del model, optimizer, scheduler, scaler, best
    gc.collect()
    if USE_AMP:
        torch.cuda.empty_cache()
    return result

## 9. Tìm siêu tham số (2 giai đoạn)

- **Giai đoạn 1:** `batch_size` và `max_len` cố định, thử `lr × weighted`.
- **Giai đoạn 2:** giữ `lr` và `weighted` tốt nhất của giai đoạn 1, thử `batch_size × max_len` (bỏ qua cấu hình đã chạy).
- Số epoch tốt nhất do early stopping quyết định (`best_epoch`).
- Mọi lần chạy được ghi vào `search_results.csv`; log từng epoch nằm trong `train_log.csv`.

In [ ]:
def expand(space):
    keys = list(space)
    return [dict(zip(keys, values)) for values in itertools.product(*(space[k] for k in keys))]


results = []
if os.path.exists(LOG_PATH):
    os.remove(LOG_PATH)


def run_stage(stage, configs):
    for hp in configs:
        hp = {k: hp[k] for k in HP_KEYS}
        if any(all(r[k] == hp[k] for k in HP_KEYS) for r in results):
            print(f"Bỏ qua cấu hình đã chạy: {hp}")
            continue
        run_id = f"{EXP_ID}_run{len(results) + 1:02d}"
        global_key = result_key(max(results, key=result_key)) if results else None
        print(f"\n=== Giai đoạn {stage} | {run_id} | {hp} ===")
        res = train_one(hp, run_id, global_key)
        results.append({"stage": stage, **res})
        pd.DataFrame(results).to_csv(os.path.join(OUT_DIR, "search_results.csv"), index=False)


stage1 = expand(SEARCH_STAGE1)
if QUICK_RUN:
    stage1 = stage1[:1]
print(f"Giai đoạn 1: {len(stage1)} cấu hình")
run_stage(1, stage1)

if not QUICK_RUN:
    best1 = max([r for r in results if r["stage"] == 1], key=result_key)
    stage2 = [{"lr": best1["lr"], "weighted": best1["weighted"], **c} for c in expand(SEARCH_STAGE2)]
    print(f"\nGiai đoạn 2: lr={best1['lr']}, weighted={best1['weighted']}")
    run_stage(2, stage2)

search_df = pd.DataFrame(results).sort_values(["val_toxic_f1", "val_macro_f1", "val_toxic_recall"], ascending=False)
display(search_df.round({c: 4 for c in search_df.columns if c.startswith("val_")}))

## 10. Đánh giá mô hình tốt nhất trên Validation

Kết quả chính dùng ngưỡng 0.5. Ngưỡng tối ưu F1 Toxic **tìm trên Validation** chỉ để tham khảo.

In [ ]:
best_run = max(results, key=result_key)
BEST_HP = {k: best_run[k] for k in HP_KEYS}
print("Cấu hình được chọn:", best_run["run_id"], BEST_HP, "| best_epoch =", best_run["best_epoch"])

best_model = AutoModelForSequenceClassification.from_pretrained(BEST_DIR).to(DEVICE)
y_val = splits["val"]["label"].to_numpy()
val_prob, _ = predict_proba(best_model, make_loader("val", BEST_HP["max_len"], EVAL_BATCH, shuffle=False))
val_metrics = compute_metrics(y_val, val_prob)

_p, _r, _t = precision_recall_curve(y_val, val_prob)
_f1 = 2 * _p[:-1] * _r[:-1] / np.clip(_p[:-1] + _r[:-1], 1e-12, None)
VAL_TUNED_THR = float(_t[np.argmax(_f1)])
val_metrics_tuned = compute_metrics(y_val, val_prob, VAL_TUNED_THR)

display(pd.DataFrame({"val @0.5": val_metrics, f"val @{VAL_TUNED_THR:.3f} (tham khảo)": val_metrics_tuned}).T)
with open(os.path.join(OUT_DIR, "val_metrics.json"), "w", encoding="utf-8") as f:
    json.dump({"thr_0.5": val_metrics, "thr_val_tuned": val_metrics_tuned}, f, ensure_ascii=False, indent=2)

# Learning curve của lần chạy tốt nhất (train loss có trọng số lớp nếu weighted=True; val loss luôn không trọng số)
log_df = pd.read_csv(LOG_PATH)
cur = log_df[log_df["run_id"] == best_run["run_id"]]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(cur["epoch"], cur["train_loss"], "o-", label="train loss")
ax[0].plot(cur["epoch"], cur["val_loss"], "o-", label="val loss")
ax[0].set_xlabel("epoch"); ax[0].set_title("Loss"); ax[0].legend()
for k in ["val_toxic_f1", "val_macro_f1", "val_toxic_recall"]:
    ax[1].plot(cur["epoch"], cur[k], "o-", label=k)
ax[1].axvline(best_run["best_epoch"], ls="--", c="gray", label="best epoch")
ax[1].set_xlabel("epoch"); ax[1].set_title("Validation"); ax[1].legend()
fig.suptitle(f"{EXP_ID} – {best_run['run_id']} {BEST_HP}")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "learning_curve.png"), dpi=150)
plt.show()

## 11. Đánh giá trên Test (một lần)

**Tập Test đã khóa:** chỉ chạy ô dưới **một lần**, sau khi đã chốt mô hình bằng Validation. Không quay lại chỉnh tham số dựa trên kết quả Test.

In [ ]:
y_test = splits["test"]["label"].to_numpy()
test_prob, _ = predict_proba(best_model, make_loader("test", BEST_HP["max_len"], EVAL_BATCH, shuffle=False))
test_metrics = compute_metrics(y_test, test_prob)
test_metrics_tuned = compute_metrics(y_test, test_prob, VAL_TUNED_THR)

display(pd.DataFrame({"test @0.5": test_metrics, f"test @{VAL_TUNED_THR:.3f} (ngưỡng từ val, tham khảo)": test_metrics_tuned}).T)
with open(os.path.join(OUT_DIR, "test_metrics.json"), "w", encoding="utf-8") as f:
    json.dump({"thr_0.5": test_metrics, "thr_val_tuned": test_metrics_tuned}, f, ensure_ascii=False, indent=2)

pred_df = pd.DataFrame({"sample_id": splits["test"]["sample_id"], "label": y_test,
                        "pred": (test_prob >= 0.5).astype(int), "prob_toxic": test_prob})
pred_df.to_csv(os.path.join(OUT_DIR, "test_predictions.csv"), index=False, encoding="utf-8-sig")

# Confusion matrix (ngưỡng 0.5)
cm = np.array([[test_metrics["tn"], test_metrics["fp"]], [test_metrics["fn"], test_metrics["tp"]]])
fig, ax = plt.subplots(figsize=(4.8, 4.2))
ax.imshow(cm, cmap="Blues")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=14,
                color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xticks([0, 1]); ax.set_xticklabels(["Non-toxic", "Toxic"])
ax.set_yticks([0, 1]); ax.set_yticklabels(["Non-toxic", "Toxic"])
ax.set_xlabel("Dự đoán"); ax.set_ylabel("Thực tế"); ax.set_title(f"{EXP_ID} – Confusion matrix (test)")
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

# Precision-Recall curve
_p, _r, _ = precision_recall_curve(y_test, test_prob)
fig, ax = plt.subplots(figsize=(5.5, 4.2))
ax.plot(_r, _p, label=f"PR-AUC = {test_metrics['pr_auc']:.4f}")
ax.axhline(y_test.mean(), ls="--", c="gray", label=f"tỉ lệ Toxic = {y_test.mean():.3f}")
ax.set_xlabel("Recall (Toxic)"); ax.set_ylabel("Precision (Toxic)"); ax.set_title(f"{EXP_ID} – Precision-Recall (test)")
ax.legend()
fig.tight_layout()
fig.savefig(os.path.join(OUT_DIR, "pr_curve.png"), dpi=150)
plt.show()

## 12. Phân tích lỗi

- **FP:** Non-toxic bị đoán là Toxic.
- **FN:** Toxic bị bỏ sót.

Các mẫu được sắp theo độ tự tin của mô hình. Toàn bộ lưu trong `errors_test.csv`.

In [ ]:
_cols = ["sample_id", "comment_type", "message_original", "message_clean"]
if "message_segmented" in splits["test"]:
    _cols.append("message_segmented")
err_df = splits["test"][_cols].copy()
err_df["label"] = y_test
err_df["pred"] = pred_df["pred"].to_numpy()
err_df["prob_toxic"] = test_prob

fp_df = err_df[(err_df["label"] == 0) & (err_df["pred"] == 1)].sort_values("prob_toxic", ascending=False).assign(error_type="FP")
fn_df = err_df[(err_df["label"] == 1) & (err_df["pred"] == 0)].sort_values("prob_toxic").assign(error_type="FN")
pd.concat([fp_df, fn_df]).to_csv(os.path.join(OUT_DIR, "errors_test.csv"), index=False, encoding="utf-8-sig")
print(f"FP: {len(fp_df)} | FN: {len(fn_df)}")

pd.set_option("display.max_colwidth", 120)
print("FP tự tin nhất:")
display(fp_df[["sample_id", "message_clean", "prob_toxic"]].head(15))
print("FN tự tin nhất:")
display(fn_df[["sample_id", "message_clean", "prob_toxic"]].head(15))

print("Kết quả trên test theo loại bình luận:")
by_type = {t: compute_metrics(g["label"], g["prob_toxic"]) for t, g in err_df.groupby("comment_type")}
display(pd.DataFrame(by_type).T[["toxic_precision", "toxic_recall", "toxic_f1", "macro_f1", "tp", "fp", "fn", "tn"]].round(4))

## 13. Lưu cấu hình và tổng kết

In [ ]:
config_out = {
    "exp_id": EXP_ID,
    "model_name": MODEL_NAME,
    "model_dir": MODEL_DIR,
    "text_col": TEXT_COL,
    "segmenter": SEGMENTER,
    "selected_hparams": BEST_HP,
    "best_run_id": best_run["run_id"],
    "best_epoch": int(best_run["best_epoch"]),
    "val_tuned_threshold": VAL_TUNED_THR,
    "selection": "val toxic_f1 (hòa: macro_f1, rồi toxic_recall); early stopping theo val toxic_f1",
    "fixed_hparams": {"seed": SEED, "max_epochs": MAX_EPOCHS, "patience": PATIENCE, "micro_batch": MICRO_BATCH,
                      "eval_batch": EVAL_BATCH, "warmup_ratio": WARMUP_RATIO, "weight_decay": WEIGHT_DECAY,
                      "max_grad_norm": MAX_GRAD_NORM, "optimizer": "AdamW", "scheduler": "linear warmup",
                      "fp16": USE_AMP, "grad_checkpointing": GRAD_CHECKPOINTING},
    "search_space": {"stage1": SEARCH_STAGE1, "stage2": SEARCH_STAGE2},
    "data": {"filter": "data_status == 'valid'",
             "paths": {"train": TRAIN_PATH, "val": VAL_PATH, "test": TEST_PATH,
                       "segmented": globals().get("SEGMENTED_PATH")},
             **{n: {"n": len(d), "toxic_ratio": round(float(d["label"].mean()), 4)} for n, d in splits.items()}},
    "quick_run": QUICK_RUN,
    "env": {"python": sys.version.split()[0], "torch": torch.__version__, "transformers": transformers.__version__,
            "scikit-learn": sklearn.__version__, "pandas": pd.__version__, "device": GPU_NAME},
}
with open(os.path.join(OUT_DIR, "config.json"), "w", encoding="utf-8") as f:
    json.dump(config_out, f, ensure_ascii=False, indent=2)

print(f"{EXP_ID} | {MODEL_NAME} | đầu vào: {TEXT_COL} | {BEST_HP} | best_epoch = {best_run['best_epoch']}")
display(pd.DataFrame({"Validation": val_metrics, "Test": test_metrics}).T[METRIC_KEYS].round(4))

print("Các file kết quả:")
for root, _, files in os.walk(OUT_DIR):
    for fn in sorted(files):
        path = os.path.join(root, fn)
        print(f"  {os.path.relpath(path, OUT_DIR):45s} {os.path.getsize(path) / 1e6:9.2f} MB")